# 08 · Publish a reproducible situation report

**40 minutes · Intermediate.** Convert the same validated inputs into an exercise sitrep, a map, a CSV and a portable briefing bundle.

[Run this lab in JupyterLite](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/lab/index.html?path=08_Situation_Reports.ipynb) · [Book home](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/book/intro.html)

**Start:** run cells from top to bottom (Shift+Enter), or use Run → Run All Cells. The first launch downloads the Python runtime and packages. Core analysis uses bundled training data; internet is needed for initial setup and interactive JavaScript. Each lab starts independently.

In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "folium==0.20.0", "plotly==6.3.0"])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from fieldguide import setup_style, load_tables, summarize, TRAINING
setup_style()
print(TRAINING)

## A useful sitrep is a dated, reviewable claim
Include the operational period, data cut-off, geographic scope, current picture, uncertainty, actions with owners, information gaps and next update. Keep what was observed separate from interpretation and proposed action. This is an original educational template informed by EOC practice; it is not an official WHO or agency form.
The generated briefing is labeled **TRAINING EXERCISE** and **DRAFT**. Use the download links to move files out of browser storage. Nothing is emailed or uploaded.

In [ ]:
from fieldguide import sitrep, district_map, report_html, download_link
s=summarize()
brief=sitrep(s)
display(Markdown(brief))

In [ ]:
fig=district_map(s)
plt.show()
portable_html=report_html(brief,fig)
display(download_link(brief,"training-sitrep.md","text/markdown"))
display(download_link(portable_html,"training-sitrep.html","text/html"))
display(download_link(s.to_csv(index=False),"training-indicators.csv","text/csv"))

## Package evidence with the narrative
The ZIP contains a self-contained HTML brief with an embedded image, Markdown, a numeric CSV, a source manifest and a map PNG. It opens without an interactive map server. A generated-at timestamp is distinct from the exercise data cut-off.

In [ ]:
import io,json,zipfile,hashlib
from datetime import datetime,timezone
from fieldguide import DATA
image_bytes=io.BytesIO()
fig.savefig(image_bytes,format="png",bbox_inches="tight")
metadata={"mode":"SYNTHETIC TRAINING", "generated_utc":datetime.now(timezone.utc).isoformat(),
          "data_window":s.attrs,"generator_seed":42,
          "input_sha256":{name:hashlib.sha256((DATA/name).read_bytes()).hexdigest()
                          for name in ["districts.csv","surveillance.csv","facilities.csv"]}}
bundle=io.BytesIO()
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    z.writestr("training-sitrep.md",brief)
    z.writestr("training-sitrep.html",portable_html)
    z.writestr("training-indicators.csv",s.to_csv(index=False))
    z.writestr("training-map.png",image_bytes.getvalue())
    z.writestr("manifest.json",json.dumps(metadata,indent=2))
display(download_link(bundle.getvalue(),"training-briefing.zip","application/zip"))
print("Bundle size:",round(len(bundle.getvalue())/1024),"KiB")

In [ ]:
# Cross-check the brief against its source table before reviewing the prose.
assert f"{s.cases.sum():,.0f}" in brief
assert s.attrs["end"] in brief
assert "TRAINING EXERCISE" in brief and "DRAFT" in brief
assert set(zipfile.ZipFile(io.BytesIO(bundle.getvalue())).namelist()) == {
    "training-sitrep.md","training-sitrep.html","training-indicators.csv","training-map.png","manifest.json"}
print("Numeric, date, exercise-label and bundle checks passed.")

## Review with a partner
Can another person reproduce the numerator and date window from the CSV? Is every map labeled? Are gaps visible? Does every proposed action have an owner and time? Read the HTML with a screen reader or without the image: the written brief and CSV should carry the key facts. Printing the HTML to PDF is available through your browser.

## Practice and debrief

Prepare a five-line executive brief from the output, keeping the training label. Add a local-language summary and a revision number. Exchange ZIP files with a classmate and ask them to find the data cut-off, denominator, main gap and next action without opening Python.

## Sources and further learning

[WHO EOC framework](https://www.who.int/publications/i/item/framework-for-a-public-health-emergency-operations-centre) · [JupyterLite file persistence](https://jupyterlite.readthedocs.io/en/stable/).